### ===========================================================================
## ESTUDIO CLÍNICO Y FACTORES DE RIESGO (Datos Crus / Raw Data)
### ===========================================================================


## 1. Carga de Datos e Configuracion Inicial

In [ ]:
import pandas as pd
import numpy as np
import os
import glob


ruta_base = '/Users/leonardoruggiero/Desktop/Proyecto Final/'
ruta_clinico = os.path.join(ruta_base, 'Estudo Clinico/')
ruta_mercado = os.path.join(ruta_base, 'ggr/')
ruta_oeda = os.path.join(ruta_base, 'oeda/')

In [ ]:
archivo_crudo = os.path.join(ruta_clinico, '2017_datos_estudio_cllinico_factores_riesgo.xlsx')
clinico = pd.read_excel(archivo_crudo, sheet_name='DATOS')
clinico.columns = clinico.columns.astype(str).str.lower().str.strip()

mapeo_columnas = {
    'p1_1': 'ano_nascimento', 'p2': 'sexo', 'p3': 'pais_nacimiento',
    'p5': 'estado_civil', 'p6': 'nivel_estudios', 'p7': 'profesion',
    'p8': 'trabaja_actualmente', 'p9': 'motivo_no_trabaja',
    'p10': 'ingresos_personales_mensuales', 'p11': 'ingresos_familiares_mensuales',
    'p12': 'recibe_ayuda_social', 'p13': 'tiene_deudas_juego_actuales',
    'p14': 'monto_deudas_juego_actuales', 'p15': 'ha_tenido_deudas_juego',
    'p16': 'monto_deudas_juego_historico', 'p27': 'fuma_actualmente', 
    'p29': 'consume_alcohol', 'p31_1': 'consume_drogas',
    'p34': 'tratamiento_salud_mental_actual', 'p35_1': 'atual_depressao',
    'p35_2': 'atual_ansiedade', 'p35_3': 'atual_fobia_especifica',
    'p35_4': 'atual_fobia_social', 'p35_5': 'atual_toc',
    'p35_6': 'atual_bipolar', 'p35_7': 'atual_psicotico',
    'p35_8': 'atual_abuso_substancias', 'p40': 'familia_historico_psicologico',
    'p44': 'gasto_maximo_diario_categoria',
    'p45': 'pais_com_problemas_jogo',
    'p46': 'frequencia_caca_as_perdas', 'p47': 'mente_sobre_ganhos',
    'p48': 'tiene_problemas_con_juego', 'p58': 'en_tratamiento_actual_por_ludopatia',
    'p59': 'tratamiento_pasado_por_ludopatia'
}
clinico.rename(columns=mapeo_columnas, inplace=True)
clinico.replace('-', np.nan, inplace=True)

mapa_pais = {1: 'España', 2: 'Unión Europea', 3: 'Resto del mundo', 9: 'No contesta'}
mapa_profesion = {
    1: 'Directivo / Gerente', 2: 'Profesional científico / intelectual', 3: 'Técnico / Profesional de apoyo', 
    4: 'Empleado administrativo', 5: 'Trabajador de servicios / ventas', 6: 'Trabajador agrícola / pesquero', 
    7: 'Artesano / Trabajador calificado', 8: 'Operador de máquinas', 9: 'Ocupaciones elementales', 
    10: 'Fuerzas armadas', 99: 'No aplica / No contesta'
}
mapa_motivo = {1: 'Estudiando', 2: 'Jubilado / Pensionista', 3: 'Labores del hogar', 4: 'Incapacidad permanente', 5: 'En paro', 6: 'Otro', 9: 'No contesta'}
mapa_frequencia = {1: 'Nunca', 2: 'A veces', 3: 'A menudo', 4: 'Casi siempre', 9: 'No responde'}
mapa_estado_civil = {1: 'Soltero/a', 2: 'Casado/a o Pareja', 3: 'Separado/a o Divorciado/a', 4: 'Viudo/a', 9: 'No contesta'}
mapa_estudios = {1: 'Sin estudios / EGB incompleto', 2: 'EGB/ESO completo', 3: 'Bachillerato inc. / FP-I', 4: 'Bachillerato comp. / FP-II', 5: 'Universidad incompleta', 6: 'Universidad completa', 7: 'Doctorado / Superior', 9: 'No desea contestar'}
mapa_gasto = {1: 'Menos de 50 €', 2: 'Entre 51 € y 100 €', 3: 'Entre 101 € y 500 €', 4: 'Entre 501 € y 1000 €', 5: 'Más de 1000 €', 6: 'No lo sabe / No recuerda', 9: 'No contesta'}
mapa_sim_no = {0: 'No', 1: 'Sí', 2: 'No', 8: 'No sabe', 9: 'No contesta'}

clinico['pais_nacimiento'] = pd.to_numeric(clinico['pais_nacimiento'], errors='coerce').map(mapa_pais)
clinico['profesion'] = pd.to_numeric(clinico['profesion'], errors='coerce').map(mapa_profesion)
clinico['motivo_no_trabaja'] = pd.to_numeric(clinico['motivo_no_trabaja'], errors='coerce').map(mapa_motivo)
clinico['frequencia_caca_as_perdas'] = pd.to_numeric(clinico['frequencia_caca_as_perdas'], errors='coerce').map(mapa_frequencia)
clinico['mente_sobre_ganhos'] = pd.to_numeric(clinico['mente_sobre_ganhos'], errors='coerce').map(mapa_frequencia)
clinico['estado_civil'] = pd.to_numeric(clinico['estado_civil'], errors='coerce').map(mapa_estado_civil)
clinico['nivel_estudios'] = pd.to_numeric(clinico['nivel_estudios'], errors='coerce').map(mapa_estudios).fillna('No especificado')
clinico['gasto_maximo_diario_categoria'] = pd.to_numeric(clinico['gasto_maximo_diario_categoria'], errors='coerce').map(mapa_gasto).fillna('No especificado')
clinico['sexo'] = pd.to_numeric(clinico['sexo'], errors='coerce').map({1: 'Masculino', 2: 'Femenino', 9: 'No desea contestar'}).fillna('No especificado')

colunas_binarias = [
    'trabaja_actualmente', 'recibe_ayuda_social', 'fuma_actualmente', 'consume_alcohol', 'consume_drogas', 
    'tratamiento_salud_mental_actual', 'atual_depressao', 'atual_ansiedade', 'atual_fobia_especifica', 
    'atual_fobia_social', 'atual_toc', 'atual_bipolar', 'atual_psicotico', 'atual_abuso_substancias', 
    'familia_historico_psicologico', 'pais_com_problemas_jogo', 'tiene_problemas_con_juego', 
    'en_tratamiento_actual_por_ludopatia', 'tratamiento_pasado_por_ludopatia'
]
for col in colunas_binarias:
    if col in clinico.columns:
        clinico[col] = pd.to_numeric(clinico[col], errors='coerce').map(mapa_sim_no)

clinico['ano_nascimento'] = pd.to_numeric(clinico['ano_nascimento'], errors='coerce')
idade_bruta = 2016 - clinico['ano_nascimento']
inval = ~idade_bruta.between(18, 90)
clinico.loc[inval, 'ano_nascimento'] = np.nan
clinico['edad_2016'] = 2016 - clinico['ano_nascimento']

clinico['faixa_etaria'] = pd.cut(
    clinico['edad_2016'], bins=[0, 25, 35, 50, 65, 200], 
    labels=['< 25', '25-34', '35-49', '50-64', '65+'], right=False
)

tiene = clinico['tiene_deudas_juego_actuales'] = pd.to_numeric(clinico['tiene_deudas_juego_actuales'], errors='coerce').map({1: 'Sí', 0: 'No', 2: 'No', 8: 'No sabe', 9: 'No contesta'})
ha = clinico['ha_tenido_deudas_juego'] = pd.to_numeric(clinico['ha_tenido_deudas_juego'], errors='coerce').map({1: 'Sí', 0: 'No', 2: 'No', 8: 'No sabe', 9: 'No contesta'})
monto = pd.to_numeric(clinico['monto_deudas_juego_actuales'], errors='coerce')

num = np.select([tiene.eq("Sí"), tiene.eq("No"), tiene.isna() & ha.eq("No")], [monto, 0, 0], default=np.nan)
clinico['monto_deudas_juego_actuales_num'] = num
clinico['nivel_endeudamiento'] = np.select(
    [pd.isna(num), num == 0, num <= 10000, num <= 50000],
    ["Sin dato", "Sin deudas", "Deuda Baja / Moderada (<= 10k€)", "Deuda Alta (10k€ - 50k€)"],
    default="Outlier / Deuda Crítica (> 50k€)"
)

clinico['ano'] = 2016
clinico['id_geografia'] = 1

colunas_finais_clinico = list(mapeo_columnas.values()) + ['edad_2016', 'faixa_etaria', 'monto_deudas_juego_actuales_num', 'nivel_endeudamiento', 'ano', 'id_geografia']
clinico = clinico[[c for c in colunas_finais_clinico if c in clinico.columns]]

/var/folders/5r/x6zcmxk907s3ljtbm1jfw1ch0000gn/T/ipykernel_37860/75685058.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clinico.replace('-', np.nan, inplace=True)
/var/folders/5r/x6zcmxk907s3ljtbm1jfw1ch0000gn/T/ipykernel_37860/75685058.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clinico['edad_2016'] = 2016 - clinico['ano_nascimento']
/var/folders/5r/x6zcmxk907s3ljtbm1jfw1ch0000gn/T/ipykernel_37860/75685058.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.ins

### ===========================================================================
## MERCADO FINANCIERO Y GGR (Datos Crus / Raw Data DGOJ)
### ===========================================================================

## 1. Importación y Lectura de las Bases Financieras de la DGOJ


In [26]:
datos_jugadores = pd.read_csv(os.path.join(ruta_mercado, 'JUEGO_ONLINE_2026.T2_NUMERO_JUGADORES.csv'), sep=None, engine='python')
datos_gastos = pd.read_csv(os.path.join(ruta_mercado, 'JUEGO_ONLINE_2026.T2_GASTOS_PROMOCION.csv'), sep=None, engine='python')
datos_depositos = pd.read_csv(os.path.join(ruta_mercado, 'JUEGO_ONLINE_2026.T2_DEPOSITOS_RETIRADAS.csv'), sep=None, engine='python')
datos_ggr = pd.read_csv(os.path.join(ruta_mercado, 'JUEGO_ONLINE_2026.T2_GGR_CANT_JUGADAS (1).csv'), sep=None, engine='python')

datos_ggr_agrupado = datos_ggr.groupby(['Año', 'Trimestre', 'Mes'], as_index=False)[['Cantidades jugadas', 'GGR']].sum()
mercado = pd.merge(datos_jugadores, datos_gastos, on=['Año', 'Trimestre', 'Mes'], how='outer')
mercado = pd.merge(mercado, datos_depositos, on=['Año', 'Trimestre', 'Mes'], how='outer')
mercado = pd.merge(mercado, datos_ggr_agrupado, on=['Año', 'Trimestre', 'Mes'], how='outer')

mercado.columns = mercado.columns.str.lower().str.replace(' ', '_').str.replace('ñ', 'n')
meses_map = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6, 'julio': 7, 'agosto': 8, 'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
mercado['mes_num'] = mercado['mes'].str.lower().map(meses_map)
mercado['data'] = pd.to_datetime(mercado['ano'].astype(str) + '-' + mercado['mes_num'].astype(str) + '-01', errors='coerce')

mercado['nº_de_depósitos'] = mercado['nº_de_depósitos'].replace(0, np.nan)
mercado['estado_n_depositos'] = np.select([mercado['nº_de_depósitos'].isna(), mercado['ano'].eq(2016)], ["ausente", "inconsistente"], default="ok")
mercado['depositos_netos'] = mercado['depósitos'] - mercado['retiradas']
mercado['ano_parcial'] = np.where(mercado.groupby("ano")["ano"].transform("size") < 12, "Si", "No")
mercado['id_geografia'] = 1

g = datos_ggr.copy()
g.columns = ['ano', 'trimestre', 'mes', 'juego', 'cantidades_jugadas', 'ggr']
g['mes_num'] = g['mes'].str.lower().map(meses_map)
g['data'] = pd.to_datetime(g['ano'].astype(str) + '-' + g['mes_num'].astype(str) + '-01', errors='coerce')

CLASE = {
    "Ap. Cruzadas": ("Apuestas", "Apuestas deportivas"), "Ap. Dep. de Contrap. Convencionales": ("Apuestas", "Apuestas deportivas"),
    "Ap. Dep. de Contrap. en Directo": ("Apuestas", "Apuestas deportivas"), "Ap. Deportivas Mutuas": ("Apuestas", "Apuestas deportivas"),
    "Ap. Hípicas de Contrapartida": ("Apuestas", "Otras apuestas"), "Apuestas Hípicas Mutuas": ("Apuestas", "Otras apuestas"),
    "Otras Ap. de Contrapartida": ("Apuestas", "Otras apuestas"), "Máquinas de Azar": ("Casino", "Slots (máquinas de azar)"),
    "Ruleta Convencional": ("Casino", "Ruleta y mesas"), "Ruleta en Vivo": ("Casino", "Ruleta y mesas"),
    "Black Jack": ("Casino", "Ruleta y mesas"), "Punto y Banca": ("Casino", "Ruleta y mesas"),
    "Juegos Complementarios": ("Casino", "Ruleta y mesas"), "Póquer Cash": ("Póquer", "Póquer"), 
    "Póquer Torneo": ("Póquer", "Póquer"), "Bingo": ("Bingo", "Bingo"), "Concursos": ("Concursos", "Concursos")
}
g["categoria_dgoj"] = g['juego'].map(lambda j: CLASE.get(j, ("Otro", "Otro"))[0])
g["segmento_historia"] = g['juego'].map(lambda j: CLASE.get(j, ("Otro", "Otro"))[1])
g['id_geografia'] = 1

### ---------------------------------------------------------------------------
##  DATOS DE SALUD PÚBLICA Y PREVALENCIA (OEDA / EDADES)
### ---------------------------------------------------------------------------

## 1. Importación de Bibliotecas y Configuración de la Carpeta OEDA


In [27]:
archivos_oeda = glob.glob(os.path.join(ruta_oeda, '*.xlsx'))
tabelas_unificadas = []

for archivo in archivos_oeda:
    nombre_base = os.path.basename(archivo).lower()
    df = pd.read_excel(archivo, header=2)
    
    if 'evolución temporal.xlsx' in archivo and 'ccaa' in nombre_base:
        continue 
        
    col_ano = [c for c in df.columns if c.lower() in ['año', 'ano', 'des_anio_encuesta']]
    ano_serie = df[col_ano[0]] if col_ano else 2024
    
    if 'mapa_porcentaje' in df.columns:
        valor_col_name = 'mapa_porcentaje'
    else:
        col_valor = [c for c in df.columns if 'porc' in c.lower() or 'indicador' in c.lower()]
        if not col_valor: continue
        valor_col_name = col_valor[0]
        
    valores_normalizados = df[valor_col_name].apply(lambda x: x / 100 if x > 1 else x)
    cols_descritivas = [c for c in df.columns if c not in col_ano and c != valor_col_name and c != 'indicador_sin_blancos']
    
    if len(cols_descritivas) > 0:
        df['Categoria'] = df[cols_descritivas].astype(str).agg(' | '.join, axis=1)
        nome_variavel = ' & '.join(cols_descritivas).title() 
    else:
        df['Categoria'] = 'Geral'
        nome_variavel = 'N/A'
        
    tema = nombre_base.replace('.xlsx', '').replace('data', 'serie').title()
    if 'comparativa por tipo de juego' in nombre_base: tema = 'Tipos De Juego'
        
    df_final = pd.DataFrame({
        'Ano': ano_serie, 'Tema_OEDA': tema, 'Eixo_Analise': nome_variavel,
        'Categoria': df['Categoria'], 'Valor_Estatistico': valores_normalizados
    })
    
    if tema == 'Tipos De Juego':
        if '2020' in nombre_base: df_final['Ano'] = 2020
        elif '2022' in nombre_base: df_final['Ano'] = 2022
        elif '2024' in nombre_base: df_final['Ano'] = 2024
        canal = 'Online' if 'online' in nombre_base else 'Presencial'
        df_final['Categoria'] = df_final['Categoria'] + f" | {canal}"

    tabelas_unificadas.append(df_final)

fact_oeda = pd.concat(tabelas_unificadas, ignore_index=True)
fact_oeda['Valor_Original'] = fact_oeda['Valor_Estatistico']

fact_oeda["es_juego"] = ~fact_oeda['Tema_OEDA'].str.contains("Internet Cius", case=False, na=False)

CORR = {
    ("Serie (6)", "Total", 2022): 0.017, ("Serie (6)", "Hombre", 2022): 0.024,
    ("Serie (6)", "Mujer", 2022): 0.009, ("Serie (6)", "Mujer", 2024): 0.007,
    ("Evolución Temporal (3)", "15-64", 2022): 0.017
}
for (tema, cat, ano), v in CORR.items():
    sel = (fact_oeda['Tema_OEDA'] == tema) & (fact_oeda['Categoria'] == cat) & (fact_oeda['Ano'] == ano)
    fact_oeda.loc[sel, "Valor_Estatistico"] = v

/Users/leonardoruggiero/Library/Python/3.9/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/leonardoruggiero/Library/Python/3.9/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/leonardoruggiero/Library/Python/3.9/lib/python/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [28]:
datas = pd.date_range(start='2013-01-01', end='2026-12-31', freq='MS')
dim_calendario = pd.DataFrame({'data': datas})
dim_calendario['ano'] = dim_calendario['data'].dt.year
dim_calendario['mes_num'] = dim_calendario['data'].dt.month
dim_calendario['trimestre'] = dim_calendario['data'].dt.quarter

comunidades = [
    'Media nacional', 'Andalucía', 'Aragón', 'Asturias, Principado de', 'Balears, Illes', 
    'Canarias', 'Cantabria', 'Castilla y León', 'Castilla - La Mancha', 'Cataluña', 
    'Comunitat Valenciana', 'Extremadura', 'Galicia', 'Madrid, Comunidad de', 
    'Murcia, Región de', 'Navarra, Comunidad Foral de', 'País Vasco', 'Rioja, La', 'Ceuta', 'Melilla'
]
dim_geografia = pd.DataFrame({'id_geografia': range(1, len(comunidades) + 1), 'nombre_region': comunidades})
dim_geografia['nivel'] = dim_geografia['nombre_region'].apply(lambda x: 'Nacional' if x == 'Media nacional' else 'Comunidad Autónoma')

nome2id = dim_geografia.set_index("nombre_region")['id_geografia'].to_dict()
nome2id.update({
    "Asturias": 4, "Illes Balears": 5, "C. Foral de Navarra": 16, "Castilla-La Mancha": 9, 
    "Comunidad Valenciana": 11, "La Rioja": 18, "Madrid": 14, "Región de Murcia": 15
})

fact_oeda['id_geografia'] = 1 
mk = fact_oeda['Categoria'].isin(nome2id.keys())
fact_oeda.loc[mk, 'id_geografia'] = fact_oeda.loc[mk, 'Categoria'].map(nome2id)

ruta_excel_final = os.path.join(ruta_base, 'Modelo_Datos_Proyecto_Final_Corrigido.xlsx')

with pd.ExcelWriter(ruta_excel_final, engine='openpyxl') as writer:
    mercado.to_excel(writer, sheet_name='Fact_MercadoFinanceiro', index=False)
    g.to_excel(writer, sheet_name='Fact_GGR_Juego', index=False)
    clinico.to_excel(writer, sheet_name='Fact_EstudioClinico', index=False)
    fact_oeda.to_excel(writer, sheet_name='Fact_OEDA', index=False)
    dim_calendario.to_excel(writer, sheet_name='Dim_Calendario', index=False)
    dim_geografia.to_excel(writer, sheet_name='Dim_Geografia', index=False)